## Dataset & Tokenizer

In [1]:
# path to dataset
DATA_PATH="../data/input.txt"

In [2]:
with open(DATA_PATH, "r", encoding= "utf-8") as file:
    data= file.read()

def find_unique_char(text: str) -> set[str]:
    return set(text)

def sort_char(text: set[str]) -> list[str]:
    return sorted(text)

using a pre-trained tokenizer, which uses Byte-Pair Encoding (BPE).

In [3]:
import tiktoken
import torch

enc= tiktoken.get_encoding("gpt2")
vocab_size= enc.n_vocab

## Deep Transformer Model

In [4]:
BATCH_SIZE= 4
BLOCK_SIZE= 64
N_EMBD= 128
N_LAYER= 4
N_HEAD= 4

In [5]:
import torch.nn as nn
import torch.nn.functional as F

class TransformerBlock(nn.Module):
    def __init__(self, N_EMBD: int, N_HEAD: int, dropout: float):
        super().__init__()
        self.layer= nn.TransformerEncoderLayer(d_model= N_EMBD, nhead= N_HEAD, dim_feedforward= 4 * N_EMBD, dropout= dropout, batch_first= True)

    def forward(self, x, src_mask):
        return self.layer(x, src_mask= src_mask)

class DeepLanguageModel(nn.Module):
    def __init__(self, vocab_size: int, n_embd: int, n_layer: int, n_head: int, block_size: int, dropout: float = 0.1):
        super().__init__()
        self.BLOCK_SIZE= BLOCK_SIZE
        self.token_embedding_table= nn.Embedding(vocab_size, N_EMBD)
        self.position_embedding_table= nn.Embedding(BLOCK_SIZE, N_EMBD)
        self.drop= nn.Dropout(dropout)

    # multiple transformer blocks
        self.blocks= nn.ModuleList([TransformerBlock(N_EMBD, N_HEAD, dropout) for _ in range(N_LAYER)])
        self.ln_f= nn.LayerNorm(N_EMBD)
        self.lm_head= nn.Linear(N_EMBD, vocab_size)

    def forward(self, idx):
        B, T= idx.shape
        tok_emb= self.token_embedding_table(idx)
        pos_emb= self.position_embedding_table(torch.arange(T, device= idx.device))

        x= self.drop(tok_emb + pos_emb)

        causal_mask= nn.Transformer.generate_square_subsequent_mask(T).to(idx.device)

        for block in self.blocks:
            x= block(x, src_mask= causal_mask)

            x= self.ln_f(x)
            return self.lm_head(x)
        
deep_model= DeepLanguageModel(vocab_size= vocab_size, n_embd= N_EMBD, n_layer= N_LAYER, n_head= N_HEAD, block_size= BLOCK_SIZE, dropout= 0.1)

In [6]:
import math

encoded_text= enc.encode(data, allowed_special= {"<|endoftext|>"})
data_tensor= torch.tensor(encoded_text, dtype= torch.long)

# split data into train and validation splits
n= int(0.9 * len(data_tensor))
train_data= data_tensor[: n]
val_data= data_tensor[n: ]

def get_batch(split, batch_size, block_size):
    data= train_data if split == "train" else val_data

    idx= torch.randint(len(data) - block_size, (block_size, ))
    x= torch.stack([data[i: i+block_size] for i in idx])
    y= torch.stack([data[i+1: i + block_size + 1] for i in idx])

    return x, y

@torch.no_grad()
def estimate_loss(model, eval_iters= 20):
    out= {}
    model.eval()

    for split in ['train', 'val']:
        losses= torch.zeros(eval_iters)
        for k in range(eval_iters):

            X, Y= get_batch(split, BATCH_SIZE, BLOCK_SIZE)
            logits= model(X)

            B, T, C= logits.shape
            loss= F.cross_entropy(logits.view(B*T, C), Y.view(B*T))
            losses[k]= loss.item()

        mean_loss= losses.mean().item()

        perplexity= math.exp(mean_loss) if mean_loss < 20 else float('inf')
        out[split]= {'loss': mean_loss, 'perplexity': perplexity}

    model.train()

    return out

In [7]:
lr_rate= 3e-4
optimizer= torch.optim.AdamW(deep_model.parameters(), lr= lr_rate)

max_steps= 500
eval_interval= 100

for step in range(max_steps):
    if step % eval_interval == 0 or step == max_steps - 1:
        metrics= estimate_loss(deep_model)
        print(f"Step {step:4d} | Train Loss: {metrics['train']['loss']:.4f} (PPL: {metrics['train']['perplexity']:8.2f}) | Val Loss: {metrics['val']['loss']:.4f} (PPL: {metrics['val']['perplexity']:8.2f})")

    xb, yb= get_batch('train', BATCH_SIZE, BLOCK_SIZE)
    logits= deep_model(xb)
    B, T, C= logits.shape
    loss= F.cross_entropy(logits.view(B * T, C), yb.view(B * T))

    optimizer.zero_grad(set_to_none= True)
    loss.backward()

    optimizer.step()



Step    0 | Train Loss: 10.9760 (PPL: 58453.81) | Val Loss: 10.9723 (PPL: 58240.14)
Step  100 | Train Loss: 6.4111 (PPL:   608.55) | Val Loss: 6.4886 (PPL:   657.60)
Step  200 | Train Loss: 6.0703 (PPL:   432.79) | Val Loss: 6.1404 (PPL:   464.22)
Step  300 | Train Loss: 5.8508 (PPL:   347.52) | Val Loss: 5.9731 (PPL:   392.74)
Step  400 | Train Loss: 5.6552 (PPL:   285.79) | Val Loss: 5.8533 (PPL:   348.39)
Step  499 | Train Loss: 5.4889 (PPL:   241.99) | Val Loss: 5.7086 (PPL:   301.46)


Watch how the perplexity (PPL) above falls as the model is trained on the dataset.

In [12]:
def generate_text(model, start_prompt: str, max_new_token: int, temperature: float= 0.8, top_k: int= 40):
    model.eval()

    tokens= enc.encode(start_prompt, allowed_special= {"<|endoftext|>"})
    idx= torch.tensor([tokens], dtype= torch.long)

    for _ in range(max_new_token):
        idx_cond= idx[:, -BLOCK_SIZE:]

        with torch.no_grad():
            logits= model(idx_cond)

        logits= logits[:, -1, :]
        logits= logits / temperature

        if top_k is not None:
            v, _= torch.topk(logits, min(top_k, logits.size(-1)))
            logits[logits < v[:, [-1]]]= -float('Inf')

        probs= F.softmax(logits, dim= -1)
        idx_next= torch.multinomial(probs, num_samples= 1)

        idx= torch.cat((idx, idx_next), dim= 1)

    return enc.decode(idx[0].tolist())

prompt = "King"
generated_output = generate_text(deep_model, start_prompt=prompt, max_new_token=100)
print(generated_output)

King
And?

A:
What; and would.

IUS:
The the world;
But of:
KING R,
B: O, so and
Thou the cause, sir, but,
And thou?
IUS:
For's, this it is
IUS:
We of I have.
A:
And-oth,

H's him of your is I have the heart that not to the world a man
